# 可选实验：逻辑回归的梯度下降

## 目标
在本实验中，你将：
- 更新用于逻辑回归的梯度下降。
- 在熟悉的数据集上探索梯度下降

In [ ]:
import copy, math
import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt
from lab_utils_common import  dlc, plot_data, plt_tumor_data, sigmoid, compute_cost_logistic
from plt_quad_logistic import plt_quad_logistic, plt_prob
plt.style.use('./deeplearning.mplstyle')

## 数据集
从决策边界实验中使用的同一个双特征数据集开始。

In [ ]:
X_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_train = np.array([0, 0, 0, 1, 1, 1])

与之前一样，我们将使用辅助函数绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，标签为 $y=0$ 的数据点显示为蓝色圆圈。

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(4,4))
plot_data(X_train, y_train, ax)

ax.axis([0, 4, 0, 3.5])
ax.set_ylabel('$x_1$', fontsize=12)
ax.set_xlabel('$x_0$', fontsize=12)
plt.show()

## 逻辑回归梯度下降
<img align="right" src="./images/C1_W3_Logistic_gradient_descent.png"     style=" width:400px; padding: 10px; " >

回顾一下，梯度下降算法使用以下梯度计算：
$$\begin{align*}
&\text{repeat until convergence:} \; \lbrace \\
&  \; \; \;w_j = w_j -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial w_j} \tag{1}  \; & \text{for j := 0..n-1} \\ 
&  \; \; \;  \; \;b = b -  \alpha \frac{\partial J(\mathbf{w},b)}{\partial b} \\
&\rbrace
\end{align*}$$

其中，每次迭代都会对所有 $j$ 同时更新 $w_j$，并且：
$$\begin{align*}
\frac{\partial J(\mathbf{w},b)}{\partial w_j}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_{j}^{(i)} \tag{2} \\
\frac{\partial J(\mathbf{w},b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)}) \tag{3} 
\end{align*}$$

* m 是数据集中的训练样本数量；
* $f_{\mathbf{w},b}(x^{(i)})$ 是模型的预测值，$y^{(i)}$ 是目标值；
* 对于逻辑回归模型：  
    $z = \mathbf{w} \cdot \mathbf{x} + b$  
    $f_{\mathbf{w},b}(x) = g(z)$  
    其中 $g(z)$ 是 sigmoid 函数：  
    $g(z) = \frac{1}{1+e^{-z}}$

### 梯度下降实现
梯度下降算法的实现包含两个部分：
- 实现上面公式 (1) 的循环，即下面的 `gradient_descent`。在可选实验和实践实验中通常会为你提供这一部分。
- 计算当前梯度，即上面的公式 (2)、(3)。这对应下面的 `compute_gradient_logistic`。本周的实践实验将要求你实现这一部分。

#### 计算梯度：代码说明
为所有 $w_j$ 和 $b$ 实现上面的公式 (2)、(3)。
实现方法有很多，下面给出其中一种思路：
- 初始化用于累加 `dj_dw` 和 `dj_db` 的变量
- 对每个样本：
    - 计算该样本的误差 $g(\mathbf{w} \cdot \mathbf{x}^{(i)} + b) - \mathbf{y}^{(i)}$
    - 对该样本中的每个输入值 $x_{j}^{(i)}$：  
        - 将误差乘以输入 $x_{j}^{(i)}$，并加到 `dj_dw` 的对应元素中。（上面的公式 2）
    - 将误差加到 `dj_db`（上面的公式 3）

- 将 `dj_db` 和 `dj_dw` 除以样本总数（m）
- 请注意，NumPy 中的 $\mathbf{x}^{(i)}$ 是 `X[i,:]` 或 `X[i]`，而 $x_{j}^{(i)}$ 是 `X[i,j]`

In [ ]:
def compute_gradient_logistic(X, y, w, b): 
    """
    Computes the gradient for linear regression 
 
    Args:
      X (ndarray (m,n): Data, m examples with n features
      y (ndarray (m,)): target values
      w (ndarray (n,)): model parameters  
      b (scalar)      : model parameter
    Returns
      dj_dw (ndarray (n,)): The gradient of the cost w.r.t. the parameters w. 
      dj_db (scalar)      : The gradient of the cost w.r.t. the parameter b. 
    """
    m,n = X.shape
    dj_dw = np.zeros((n,))                           #(n,)
    dj_db = 0.

    for i in range(m):
        f_wb_i = sigmoid(np.dot(X[i],w) + b)          #(n,)(n,)=scalar
        err_i  = f_wb_i  - y[i]                       #scalar
        for j in range(n):
            dj_dw[j] = dj_dw[j] + err_i * X[i,j]      #scalar
        dj_db = dj_db + err_i
    dj_dw = dj_dw/m                                   #(n,)
    dj_db = dj_db/m                                   #scalar
        
    return dj_db, dj_dw  

使用下面的单元格检查梯度函数的实现。

In [ ]:
X_tmp = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])
y_tmp = np.array([0, 0, 0, 1, 1, 1])
w_tmp = np.array([2.,3.])
b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_tmp, y_tmp, w_tmp, b_tmp)
print(f"dj_db: {dj_db_tmp}" )
print(f"dj_dw: {dj_dw_tmp.tolist()}" )

**预期输出**
``` 
dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]
```

#### 梯度下降代码
下面实现了上面的方程 (1)。请花一点时间在例程中找到各个函数，并将它们与上面的方程进行比较。

In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters): 
    """
    Performs batch gradient descent
    
    Args:
      X (ndarray (m,n)   : Data, m examples with n features
      y (ndarray (m,))   : target values
      w_in (ndarray (n,)): Initial values of model parameters  
      b_in (scalar)      : Initial values of model parameter
      alpha (float)      : Learning rate
      num_iters (scalar) : number of iterations to run gradient descent
      
    Returns:
      w (ndarray (n,))   : Updated values of parameters
      b (scalar)         : Updated value of parameter 
    """
    # An array to store cost J and w's at each iteration primarily for graphing later
    J_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):
        # Calculate the gradient and update the parameters
        dj_db, dj_dw = compute_gradient_logistic(X, y, w, b)   

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               
      
        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            J_history.append( compute_cost_logistic(X, y, w, b) )

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters / 10) == 0:
            print(f"Iteration {i:4d}: Cost {J_history[-1]}   ")
        
    return w, b, J_history         #return final w,b and J history for graphing


让我们在数据集上运行梯度下降。

In [ ]:
w_tmp  = np.zeros_like(X_train[0])
b_tmp  = 0.
alph = 0.1
iters = 10000

w_out, b_out, _ = gradient_descent(X_train, y_train, w_tmp, b_tmp, alph, iters) 
print(f"\nupdated parameters: w:{w_out}, b:{b_out}")

#### 绘制梯度下降的结果：

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(5,4))
# plot the probability 
plt_prob(ax, w_out, b_out)

# Plot the original data
ax.set_ylabel(r'$x_1$')
ax.set_xlabel(r'$x_0$')   
ax.axis([0, 4, 0, 3.5])
plot_data(X_train,y_train,ax)

# Plot the decision boundary
x0 = -b_out/w_out[1]
x1 = -b_out/w_out[0]
ax.plot([0,x0],[x1,0], c=dlc["dlblue"], lw=1)
plt.show()

在上图中：
 - 阴影表示 y=1 的概率（应用决策边界之前的结果）；
 - 决策边界是概率等于 0.5 的线。
 

## 另一个数据集
让我们回到单变量数据集。由于只有 $w$、$b$ 两个参数，可以使用等高线图绘制代价函数，从而更好地了解梯度下降的行为。

In [ ]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

与之前一样，我们将使用辅助函数绘制这些数据。标签为 $y=1$ 的数据点显示为红色叉号，标签为 $y=0$ 的数据点显示为黑色圆圈。

In [ ]:
fig,ax = plt.subplots(1,1,figsize=(4,3))
plt_tumor_data(x_train, y_train, ax)
plt.show()

在下图中，请尝试：
- 在右上角的等高线图中单击，以改变 $w$ 和 $b$；
    - 更改可能需要一两秒才会生效；
    - 注意左上图中不断变化的代价值；
    - 注意，总代价由每个样本的损失累积而成（竖直虚线）；
- 单击橙色按钮运行梯度下降；
    - 注意代价持续下降（等高线图和代价图使用 log(cost)）；
    - 单击等高线图会重置模型，以便重新运行；
- 若要重置图表，请重新运行该单元格。

In [ ]:
w_range = np.array([-1, 7])
b_range = np.array([1, -14])
quad = plt_quad_logistic( x_train, y_train, w_range, b_range )

## 恭喜！
你已经：
- 研究了逻辑回归梯度计算的公式和实现
- 将这些例程用于
    - 探索单变量数据集
    - 探索双变量数据集